# google-ai-generativelanguage smoke test

Smoke test for `google-ai-generativelanguage`, the generated client for the Gemini API. Every example
runs offline: the client is built with anonymous credentials, and requests and responses are built
locally and round-tripped. No request is sent to the service.

## Example 1: Client construction and resource paths

In [ ]:
from google.ai import generativelanguage as genai
from google.auth.credentials import AnonymousCredentials

# Example 1: construct the client offline and use its resource-path helpers.
# Building a client opens no connection; RPCs only start when a method is called.
client = genai.GenerativeServiceClient(credentials=AnonymousCredentials())
print("transport:", type(client.transport).__name__)
print("endpoint:", client.api_endpoint)
assert client.api_endpoint == "generativelanguage.googleapis.com"
assert callable(client.generate_content) and callable(client.count_tokens)

Client = genai.GenerativeServiceClient
model_name = Client.model_path("demo-model")
assert model_name == "models/demo-model", model_name
assert Client.parse_model_path(model_name) == {"model": "demo-model"}
print("model path:", model_name)

## Example 2: Multi-turn content request and wire-format round trip

In [ ]:
from google.ai import generativelanguage as genai

# Example 2: build a multi-turn GenerateContentRequest and round-trip it through the wire format.
request = genai.GenerateContentRequest(
    model="models/demo-model",
    contents=[
        genai.Content(role="user", parts=[genai.Part(text="What is 2 + 2?")]),
        genai.Content(role="model", parts=[genai.Part(text="4")]),
        genai.Content(role="user", parts=[genai.Part(text="And doubled?")]),
    ],
)
assert len(request.contents) == 3
assert [c.role for c in request.contents] == ["user", "model", "user"]

wire = genai.GenerateContentRequest.serialize(request)
print(f"serialized request: {len(wire)} bytes")
decoded = genai.GenerateContentRequest.deserialize(wire)
assert decoded == request
assert decoded.contents[2].parts[0].text == "And doubled?"
assert decoded.model == "models/demo-model"

## Example 3: Generation config and safety settings as JSON and dict

In [ ]:
from google.ai import generativelanguage as genai

# Example 3: generation and safety settings, as JSON and as a dict.
config = genai.GenerationConfig(
    temperature=0.25, top_k=40, max_output_tokens=256, candidate_count=1, stop_sequences=["END"],
)
safety = genai.SafetySetting(
    category=genai.HarmCategory.HARM_CATEGORY_HARASSMENT,
    threshold=genai.SafetySetting.HarmBlockThreshold.BLOCK_ONLY_HIGH,
)
request = genai.GenerateContentRequest(
    model="models/demo-model",
    contents=[genai.Content(role="user", parts=[genai.Part(text="hi")])],
    generation_config=config,
    safety_settings=[safety],
)

as_json = genai.GenerateContentRequest.to_json(request, use_integers_for_enums=False, indent=None)
print(f"JSON: {len(as_json)} characters")
assert '"HARM_CATEGORY_HARASSMENT"' in as_json and '"BLOCK_ONLY_HIGH"' in as_json  # enums by name
assert '"maxOutputTokens": 256' in as_json  # JSON uses camelCase field names
assert genai.GenerateContentRequest.from_json(as_json) == request

as_dict = genai.GenerateContentRequest.to_dict(request)
assert as_dict["generation_config"]["max_output_tokens"] == 256
assert as_dict["generation_config"]["stop_sequences"] == ["END"]
assert abs(as_dict["generation_config"]["temperature"] - 0.25) < 1e-6
# to_dict keeps the proto field names and writes enums as their integer values.
assert as_dict["safety_settings"][0]["threshold"] == genai.SafetySetting.HarmBlockThreshold.BLOCK_ONLY_HIGH
print("generation_config:", {k: as_dict["generation_config"][k] for k in ("top_k", "max_output_tokens")})

## Example 4: Embedding, token-counting and inline binary data

In [ ]:
from google.ai import generativelanguage as genai

# Example 4: embedding and token-counting requests, plus inline binary data.
embed = genai.EmbedContentRequest(
    model="models/demo-embedding",
    content=genai.Content(parts=[genai.Part(text="a sentence to embed")]),
    task_type=genai.TaskType.RETRIEVAL_DOCUMENT,
    title="Demo document",
)
batch = genai.BatchEmbedContentsRequest(model="models/demo-embedding", requests=[embed, embed])
assert len(batch.requests) == 2
assert batch.requests[0].task_type == genai.TaskType.RETRIEVAL_DOCUMENT
assert genai.BatchEmbedContentsRequest.deserialize(genai.BatchEmbedContentsRequest.serialize(batch)) == batch

# Binary parts carry raw bytes and a MIME type.
image_part = genai.Part(inline_data=genai.Blob(mime_type="image/png", data=b"\x89PNG\r\n\x1a\n"))
count = genai.CountTokensRequest(
    model="models/demo-model",
    contents=[genai.Content(role="user", parts=[genai.Part(text="describe this"), image_part])],
)
round_tripped = genai.CountTokensRequest.deserialize(genai.CountTokensRequest.serialize(count))
blob = round_tripped.contents[0].parts[1].inline_data
assert (blob.mime_type, blob.data) == ("image/png", b"\x89PNG\r\n\x1a\n")
print("embed batch:", len(batch.requests), "requests | count_tokens parts:", len(round_tripped.contents[0].parts))

## Example 5: Function declarations and function-call responses

In [ ]:
from google.ai import generativelanguage as genai

# Example 5: function calling: declare a tool, then read a function call back from a response.
weather = genai.FunctionDeclaration(
    name="get_weather",
    description="Current weather for a city",
    parameters=genai.Schema(
        type_=genai.Type.OBJECT,
        properties={"city": genai.Schema(type_=genai.Type.STRING)},
        required=["city"],
    ),
)
tool = genai.Tool(function_declarations=[weather])
assert tool.function_declarations[0].parameters.properties["city"].type_ == genai.Type.STRING

# A response as the service would return it, built locally: one candidate that calls the tool.
response = genai.GenerateContentResponse(candidates=[genai.Candidate(
    content=genai.Content(role="model", parts=[genai.Part(
        function_call=genai.FunctionCall(name="get_weather", args={"city": "Paris"}))]),
    finish_reason=genai.Candidate.FinishReason.STOP,
)])
decoded = genai.GenerateContentResponse.from_json(genai.GenerateContentResponse.to_json(response))
call = decoded.candidates[0].content.parts[0].function_call
assert call.name == "get_weather" and dict(call.args) == {"city": "Paris"}
assert decoded.candidates[0].finish_reason == genai.Candidate.FinishReason.STOP
print("function call:", call.name, dict(call.args))